# 03 — Agent Testing

Test the Cash Agent locally using ADK and deploy to Vertex AI Agent Engine.

In [ ]:
import os
os.environ['PROJECT_ID'] = 'your-project-id'
os.environ['DATASET_ID'] = 'cash_agent_demo'
os.environ['MODEL'] = 'gemini-2.5-pro'
os.environ['FLASH_MODEL'] = 'gemini-2.5-flash'

## Test Locally with ADK Runner

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

from agent import root_agent

session_service = InMemorySessionService()
runner = Runner(
    agent=root_agent,
    app_name='cash_agent_test',
    session_service=session_service,
)

In [ ]:
async def ask(message: str, user_id='test_user', session_id='test_session'):
    """Send a message to the agent and print the response."""
    content = types.Content(
        role='user',
        parts=[types.Part.from_text(message)]
    )
    print(f"\n>>> {message}\n")
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=content,
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text)

## Scene 1: Morning Cash Position Check

In [ ]:
await ask("Good morning. What's our current cash position across all currencies?")

## Scene 2: Cash Flow Forecast

In [ ]:
await ask("What does our cash flow look like for the next 30 days? Break it down by currency.")

## Scene 3: Anomaly Detection

In [ ]:
await ask("Are there any unusual patterns or risks I should know about?")

## Scene 4: Recommendations

In [ ]:
await ask("Given all of this, what do you recommend we do?")

## Scene 5: Execute with Approval

In [ ]:
await ask("Go ahead with recommendation 1 - place the EUR deposit.")

## Scene 6: Execute FX Hedge

In [ ]:
await ask("Now execute the GBP hedge - recommendation 2.")

## Scene 7: What-If Scenario

In [ ]:
await ask("What if ACME Corp doesn't pay and the EUR weakens by 5% against USD?")

## Deploy to Vertex AI Agent Engine

In [ ]:
import vertexai
from vertexai import agent_engines
from vertexai.preview import reasoning_engines

PROJECT_ID = os.environ['PROJECT_ID']
STAGING_BUCKET = f'gs://{PROJECT_ID}-cash-agent-staging'

vertexai.init(project=PROJECT_ID, location='us-central1', staging_bucket=STAGING_BUCKET)

In [ ]:
# Test with AdkApp wrapper first
app = reasoning_engines.AdkApp(agent=root_agent, enable_tracing=True)

session = app.create_session(user_id='test_user')
print(f"Session: {session}")

for event in app.stream_query(
    user_id='test_user',
    session_id=session['id'],
    message="What's our cash position?",
):
    print(event)

In [ ]:
# Deploy to Agent Engine
remote_app = agent_engines.create(
    agent_engine=root_agent,
    requirements=[
        'google-cloud-aiplatform[adk,agent_engines]',
        'google-cloud-bigquery',
        'google-cloud-storage',
        'requests',
    ],
    display_name='Cash Agent Demo',
    description='AI-powered Treasury Cash Agent',
)

print(f"Deployed! Resource name: {remote_app.resource_name}")
print(f"Agent Engine ID: {remote_app.resource_name.split('/')[-1]}")